In [1]:
from utils import pr
import numpy as np
import pandas as pd

In [2]:
users = pd.DataFrame(
    {
        "user_id": ["001", "002", "003", "004"],
        "signup_data": pd.to_datetime(
            ["2023-01-01", "2023-01-05", "2023-01-10", "2023-01-15"]
        ),
        "churned": [False, True, False, True],
    }
)
transactions = pd.DataFrame(
    {
        "user_id": ["001", "001", "002", "003", "003", "004"],
        "transaction_data": pd.to_datetime(
            [
                "2023-01-03",
                "2023-02-10",  # AFTER churn for some users
                "2023-01-06",
                "2023-01-12",
                "2023-03-01",  # AFTER churn
                "2023-02-20",  # AFTER churn
            ]
        ),
        "amount": [100, 500, 50, 200, 1000, 300],
    }
)
pr(users , 'users')
pr(transactions , 'transactions')

+++++++++++++++++ users ++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,signup_data,churned
0,001,2023-01-01,False
1,002,2023-01-05,True
2,003,2023-01-10,False
3,004,2023-01-15,True


-----

+++++++++++++++++ transactions ++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,transaction_data,amount
0,001,2023-01-03,100
1,001,2023-02-10,500
2,002,2023-01-06,50
3,003,2023-01-12,200
4,003,2023-03-01,1000
5,004,2023-02-20,300


-----



In [3]:
merged = users.merge(transactions , on='user_id' , how='left')
pr(merged , 'merged')

+++++++++++++++++ merged ++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,signup_data,churned,transaction_data,amount
0,001,2023-01-01,False,2023-01-03,100
1,001,2023-01-01,False,2023-02-10,500
2,002,2023-01-05,True,2023-01-06,50
3,003,2023-01-10,False,2023-01-12,200
4,003,2023-01-10,False,2023-03-01,1000
5,004,2023-01-15,True,2023-02-20,300


-----



In [21]:
total_spend = transactions.groupby('user_id')['amount'].sum().reset_index().rename({'amount': 'total'} , axis=1)
users_with_spend = transactions.merge(total_spend , on='user_id' , how='left')
pr(total_spend)
pr(total_spend.dtypes)
pr(users_with_spend)

+++++++++++++++++++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,total
0,001,600
1,002,50
2,003,1200
3,004,300


-----

+++++++++++++++++++++++++++++++++++
<class 'pandas.Series'>


user_id      str
total      int64
dtype: object

-----

+++++++++++++++++++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,transaction_data,amount,total
0,001,2023-01-03,100,600
1,001,2023-02-10,500,600
2,002,2023-01-06,50,50
3,003,2023-01-12,200,1200
4,003,2023-03-01,1000,1200
5,004,2023-02-20,300,300


-----



In [23]:
merged = users.merge(transactions , on='user_id' , how='left')
mask = merged['transaction_data'] <= merged['signup_data'] + pd.Timedelta(days=7)
pr(mask , 'mask')
merged = merged[mask]
pr(merged)

+++++++++++++++++ mask ++++++++++++++++++
<class 'pandas.Series'>


0     True
1    False
2     True
3     True
4    False
5    False
dtype: bool

-----

+++++++++++++++++++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,signup_data,churned,transaction_data,amount
0,001,2023-01-01,False,2023-01-03,100
2,002,2023-01-05,True,2023-01-06,50
3,003,2023-01-10,False,2023-01-12,200


-----

